# 🎯 Análise de Centralidade - Game of Thrones
## Quem é matematicamente o personagem mais importante?

In [1]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import numpy as np

In [2]:
df = pd.read_csv("../datasets/interacoes.csv")
df_direct = df#[df["tipo_interacao"] == "single"]
df_grouped = df_direct.groupby(['falante', 'ouvinte']).size().reset_index(name='weight')
G = nx.from_pandas_edgelist(df_grouped, 'falante', 'ouvinte', edge_attr='weight', create_using=nx.Graph())
print(f"✅ Grafo: {G.number_of_nodes()} nós, {G.number_of_edges()} arestas")

✅ Grafo: 138 nós, 1091 arestas


## 1️⃣ Centralidade de Grau

In [3]:
degree_cent = nx.degree_centrality(G)
top_degree = sorted(degree_cent.items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(top_degree, columns=['Personagem', 'Score'])

,Personagem,Score
0,JOFFREY BARATHEON,0.518248
1,TYRION LANNISTER,0.452555
2,CATELYN STARK,0.408759
3,BRIENNE DE TARTH,0.401460
4,JAIME LANNISTER,0.357664
5,DAENERYS TARGARYEN,0.343066
6,THEON GREYJOY,0.313869
7,DAISY,0.306569
8,SANSA STARK,0.291971
9,BOWEN MARSH,0.277372


## 2️⃣ Centralidade de Intermediação

In [4]:
betweenness_cent = nx.betweenness_centrality(G, weight='weight')
top_betweenness = sorted(betweenness_cent.items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(top_betweenness, columns=['Personagem', 'Score'])

,Personagem,Score
0,BRIENNE DE TARTH,0.215457
1,JOFFREY BARATHEON,0.122996
2,DAISY,0.108345
3,JAIME LANNISTER,0.071066
4,WALDA BOLTON,0.068387
5,CAMELLO,0.065370
6,DAENERYS TARGARYEN,0.065264
7,GREY WORM,0.063426
8,AXELL FLORENT,0.055914
9,PODRICK PAYNE,0.052215


## 3️⃣ PageRank

In [5]:
pagerank = nx.pagerank(G, weight='weight', max_iter=100, tol=1e-06)
top_pagerank = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(top_pagerank, columns=['Personagem', 'Score'])

,Personagem,Score
0,TYRION LANNISTER,0.059793
1,JOFFREY BARATHEON,0.052936
2,CATELYN STARK,0.037560
3,DAENERYS TARGARYEN,0.036395
4,JAIME LANNISTER,0.029741
5,SANSA STARK,0.028055
6,BRIENNE DE TARTH,0.027030
7,ARYA STARK,0.026170
8,THEON GREYJOY,0.025752
9,SAMWELL TARLY,0.024931


## 4️⃣ Closeness Centrality

In [6]:
closeness_centrality = nx.closeness_centrality(G, distance='weight')
top_closeness = sorted(closeness_centrality.items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(top_closeness, columns=['Personagem', 'Score'])

,Personagem,Score
0,BRIENNE DE TARTH,0.332530
1,CAMELLO,0.300684
2,WALDA BOLTON,0.300684
3,JOFFREY BARATHEON,0.300016
4,DAISY,0.297373
5,DAENERYS TARGARYEN,0.294134
6,MAESTER WOLKAN,0.293494
7,MATTHOS SEAWORTH,0.291592
8,GREY WORM,0.289715
9,XARO XHOAN DAXOS,0.288477


## 5️⃣ Weighted Degree

In [7]:
weighted_degree = dict(G.degree(weight='weight'))
top_weighted = sorted(weighted_degree.items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(top_weighted, columns=['Personagem', 'Interações'])

,Personagem,Interações
0,TYRION LANNISTER,2614
1,JOFFREY BARATHEON,2129
2,CATELYN STARK,1476
3,DAENERYS TARGARYEN,1473
4,JAIME LANNISTER,1250
5,BRIENNE DE TARTH,1140
6,SANSA STARK,1131
7,ARYA STARK,1046
8,THEON GREYJOY,898
9,SAMWELL TARLY,857


## 🏆 Ranking Consolidado
**Pesos:** PageRank(30%), Betweenness(25%), Degree(25%), Closeness(20%)

In [8]:
personagens = list(degree_cent.keys())
scaler = MinMaxScaler()

norm_pr = scaler.fit_transform(np.array([pagerank[p] for p in personagens]).reshape(-1,1)).flatten()
norm_bt = scaler.fit_transform(np.array([betweenness_cent[p] for p in personagens]).reshape(-1,1)).flatten()
norm_dc = scaler.fit_transform(np.array([degree_cent[p] for p in personagens]).reshape(-1,1)).flatten()
norm_cc = scaler.fit_transform(np.array([closeness_centrality[p] for p in personagens]).reshape(-1,1)).flatten()

scores = {p: norm_pr[i]*0.3 + norm_bt[i]*0.25 + norm_dc[i]*0.25 + norm_cc[i]*0.2 for i,p in enumerate(personagens)}
top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:20]

df_final = pd.DataFrame([
    (p, scores[p], pagerank[p], betweenness_cent[p], degree_cent[p], closeness_centrality[p]) 
    for p,_ in top
], columns=["Personagem","Score","PageRank","Betweenness","Degree","Closeness"])

df_final

,Personagem,Score,PageRank,Betweenness,Degree,Closeness
0,JOFFREY BARATHEON,0.838062,0.052936,0.122996,0.518248,0.300016
1,BRIENNE DE TARTH,0.774341,0.027030,0.215457,0.401460,0.332530
2,TYRION LANNISTER,0.724287,0.059793,0.041361,0.452555,0.264202
3,DAENERYS TARGARYEN,0.595922,0.036395,0.065264,0.343066,0.294134
4,CATELYN STARK,0.576952,0.037560,0.032778,0.408759,0.261136
5,JAIME LANNISTER,0.570246,0.029741,0.071066,0.357664,0.284826
6,DAISY,0.551879,0.021182,0.108345,0.306569,0.297373
7,THEON GREYJOY,0.481031,0.025752,0.037056,0.313869,0.272192
8,SANSA STARK,0.466840,0.028055,0.032334,0.291971,0.256181
9,ARYA STARK,0.465431,0.026170,0.044729,0.255474,0.276089
